In [1]:
from ts_inverse import datahandler
import ts_inverse.models as models
from ts_inverse.models.fcn import FCN_Predictor
import ts_inverse.utils as utils
import ts_inverse.server as server
import ts_inverse.client as client
import flwr as fl
import torch
import copy
from tqdm import tqdm
from collections import OrderedDict

NUM_CLIENTS = 10
FL_ROUNDS = 50
DEVICE = 'cuda'
DATASET_CONFIG = {
    'dataset': 'london_smartmeter',
    'train_stride': 1,
    'validation_stride': 24,
    'observation_days': 1,
    'future_days': 1,
    'normalize': 'minmax',
}

MODEL_CONFIG = {
        '_model': FCN_Predictor,
        'hidden_size': 64,
        '_attack_step_multiplier': 1,
}

trainsets, valsets, testsets = datahandler.get_datasets(**DATASET_CONFIG, columns=0)
trainset, valset, testset = trainsets[0], valsets[0], testsets[0]

client_resources = {"num_cpus": 1, "num_gpus": 0.1}

model = MODEL_CONFIG['_model'](features=[0], hidden_size=MODEL_CONFIG['hidden_size'], input_size=trainset.freq_in_day*DATASET_CONFIG['observation_days'], output_size=trainset.freq_in_day*DATASET_CONFIG['future_days'])
model_parameters = utils.get_model_parameters(model)

# Create strategy
strategy = server.CustomStrategy(
    on_fit_config_fn=server.fit_config,
    evaluate_metrics_aggregation_fn=server.evaluate_metrics_aggregation,
    initial_parameters=fl.common.ndarrays_to_parameters(model_parameters),
)

/scratch/ejk5818/miniconda3/envs/ts-inverse/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
history = fl.simulation.start_simulation(
    client_fn=client.client_factory(d_config=DATASET_CONFIG, m_config=MODEL_CONFIG, device=DEVICE, num_clients=NUM_CLIENTS),
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=FL_ROUNDS),  
    strategy=strategy,
    client_resources=client_resources,
    ray_init_args={"num_cpus": 64, "num_gpus": 1},
)

2025-12-21 23:11:05,405	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Number of GPUs available: 8
Loading dataset for 10 clients: {'dataset': 'london_smartmeter', 'train_stride': 1, 'validation_stride': 24, 'observation_days': 1, 'future_days': 1, 'normalize': 'minmax'}


	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=50, no round_timeout


Finished loading datasets.


2025-12-21 23:12:01,309	INFO worker.py:2012 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'accelerator_type:RTX': 1.0, 'node:10.136.74.9': 1.0, 'object_store_memory': 150009956352.0, 'GPU': 1.0, 'memory': 350023231488.0, 'CPU': 64.0, 'node:__internal_head__': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.1}
INFO :      Flower VCE: Creating VirtualClientEngineActorPool with 10 actors
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(ClientAppActor pid=2866534) /scratch/ejk5818/miniconda3/envs/ts-inverse/lib/python3.11/site-packages/torch/

...

INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)


Client 1087403622438670090 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
Client 16197780566663339875 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
Client 8544152730744657721 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
Client 5137690552222161226 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
Client 6704707500307356003 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
Client 18351099303203063569 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
Client 14381246421098101976 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
Client 11644052494521541154 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
Client 16566188917919038450 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
Client 15798156788968614209 weights shapes: [(64, 48), (64,), (64, 64), (64,), (48, 64), (48,)]
(ClientAppActor pid=2866527) Loading client 

INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 50 round(s) in 2557.62s
INFO :      	History (loss, distributed):
INFO :      		round 1: 0.01469601903601821
INFO :      		round 2: 0.014364072675102157
INFO :      		round 3: 0.014389010217236441
INFO :      		round 4: 0.01377293216131712
INFO :      		round 5: 0.013917410982164615
INFO :      		round 6: 0.014374777150985878
INFO :      		round 7: 0.014283515877415541
INFO :      		round 8: 0.014101403449824124
INFO :      		round 9: 0.013650889432654321
INFO :      		round 10: 0.013702860593615458
INFO :      		round 11: 0.01344934099974412
INFO :      		round 12: 0.013118743699861175
INFO :      		round 13: 0.012935190962245793
INFO :      		round 14: 0.012493115341929617
INFO :      		round 15: 0.012215360063468668
INFO :      		round 16: 0.011885997973551282
INFO :      		round 17: 0.011521754156698373
INFO :      		round 18: 0.011302278309976098
INFO :   

Evaluate Metrics: [(101, {'mae': 0.04671813175082207, 'mse': 0.004323807545006275, 'rmse': 0.06575566549740239, 'smape': 0.9392125606536865}), (101, {'mae': 0.06969234347343445, 'mse': 0.012482846155762672, 'rmse': 0.11172665821442379, 'smape': 0.4487127661705017}), (79, {'mae': 0.0873485654592514, 'mse': 0.015296172350645065, 'rmse': 0.12367769544523809, 'smape': 0.414731502532959}), (103, {'mae': 0.05165410414338112, 'mse': 0.006123762112110853, 'rmse': 0.07825447023723854, 'smape': 0.5748935341835022}), (85, {'mae': 0.05287248641252518, 'mse': 0.006830602418631315, 'rmse': 0.08264745863383408, 'smape': 0.6702689528465271}), (97, {'mae': 0.05287204682826996, 'mse': 0.0067429132759571075, 'rmse': 0.08211524387077657, 'smape': 0.7873533964157104}), (160, {'mae': 0.08762522041797638, 'mse': 0.01669662818312645, 'rmse': 0.12921543322346, 'smape': 0.8561165928840637}), (100, {'mae': 0.05221553519368172, 'mse': 0.00714259734377265, 'rmse': 0.08451388846676415, 'smape': 0.6586865186691284})

INFO :      	           0.06348788738250732,
INFO :      	           0.0574658140540123,
INFO :      	           0.08455970138311386,
INFO :      	           0.0637415274977684,
INFO :      	           0.06435804814100266]),
INFO :      	         (14,
INFO :      	          [0.058325376361608505,
INFO :      	           0.06390731781721115,
INFO :      	           0.06104157865047455,
INFO :      	           0.05595025047659874,
INFO :      	           0.0693352147936821,
INFO :      	           0.06297910958528519,
INFO :      	           0.08346197009086609,
INFO :      	           0.11213313043117523,
INFO :      	           0.08410540223121643,
INFO :      	           0.09677615761756897]),
INFO :      	         (15,
INFO :      	          [0.06228860840201378,
INFO :      	           0.06271946430206299,
INFO :      	           0.10909711569547653,
INFO :      	           0.06091465428471565,
INFO :      	           0.08417128771543503,
INFO :      	           0.0566834881901741,


In [4]:
import pickle
pickle.dump(history.metrics_distributed, open("./federated_learning_history_metrics_distributed.pkl", "wb"))

In [6]:
metrics_distributed = pickle.load(open("./federated_learning_history_metrics_distributed.pkl", "rb"))

In [6]:
global_model = MODEL_CONFIG['_model'](features=[0], hidden_size=MODEL_CONFIG['hidden_size'], input_size=trainset.freq_in_day*DATASET_CONFIG['observation_days'], output_size=trainset.freq_in_day*DATASET_CONFIG['future_days'])
global_model.to(DEVICE)
params_dict = zip(global_model.state_dict().keys(), strategy.weight)
state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
global_model.load_state_dict(state_dict, strict=True)

RuntimeError: Could not infer dtype of Parameters

In [9]:
def get_dataset_for_client(cid, d_config):
    cid = int(cid)
    config = copy.deepcopy(d_config)
    del config['dataset']
    # print(f"Getting dataset for client {cid} with config: {config}")
    dataset_df, dataset_class = datahandler.get_dataset_df(d_config["dataset"])
    return datahandler.get_datasets_from_df(dataset_df, dataset_class, cid, **config)

In [9]:
history.metrics_distributed

{'mae': [(1, 0.0813129219685966)]}

In [11]:
# regenerate the clients
from ts_inverse.client import TimeSeriesClient
DATASET_CONFIG = {
    'dataset': 'london_smartmeter',
    'train_stride': 1,
    'validation_stride': 24,
    'observation_days': 1,
    'future_days': 1,
    'normalize': 'minmax',
}
MODEL_CONFIG = {
        '_model': FCN_Predictor,
        'hidden_size': 64,
        '_attack_step_multiplier': 1,
}
clients = [TimeSeriesClient(model_config=MODEL_CONFIG, dataset_config={**DATASET_CONFIG, "columns": i}, datasets=get_dataset_for_client(i, DATASET_CONFIG), device=DEVICE) for i in range(NUM_CLIENTS)]

In [ ]:
private_model_eval_dict = {}

for e in range(FL_ROUNDS):
    private_model_eval_dict[e] = {}
    for i, c in enumerate(clients):
        params, _, _ = c.fit(c.get_parameters(), {"lr": 0.01, "batch_size": 1, "epochs": 1})
        c.set_parameters(params)
        eval_dict = c.evaluate(c.get_parameters(), {})
        private_model_eval_dict[e][i] = eval_dict

    print(f"Completed evaluation for round {e}")
    print(private_model_eval_dict[e])

Completed evaluation for round 0
{0: (0.009932311251759529, 97, {'mae': 0.06366323679685593, 'mse': 0.009932311251759529, 'rmse': 0.09966098159139077, 'smape': 0.9408841133117676}), 1: (0.020806217566132545, 160, {'mae': 0.10516440868377686, 'mse': 0.020806217566132545, 'rmse': 0.1442436049401586, 'smape': 0.9722421169281006}), 2: (0.019909337162971497, 79, {'mae': 0.11578308045864105, 'mse': 0.019909337162971497, 'rmse': 0.14110045061221987, 'smape': 0.5520818829536438}), 3: (0.017775816842913628, 42, {'mae': 0.08614540845155716, 'mse': 0.017775816842913628, 'rmse': 0.13332597962480391, 'smape': 0.7279548645019531}), 4: (0.00999084860086441, 103, {'mae': 0.06578930467367172, 'mse': 0.00999084860086441, 'rmse': 0.09995423253101597, 'smape': 0.8188474178314209}), 5: (0.008567185141146183, 103, {'mae': 0.07112009823322296, 'mse': 0.008567185141146183, 'rmse': 0.0925590899973967, 'smape': 0.7770609855651855}), 6: (0.007999622263014317, 100, {'mae': 0.05863863229751587, 'mse': 0.0079996222

In [45]:
private_model_eval_dict = {}

for i, c in enumerate(clients):
    eval_dict = c.evaluate(c.get_parameters(), {})
    private_model_eval_dict[i] = eval_dict[2]

In [47]:
eval_dict

(0.010006822645664215, 85, {'mae': 0.06849749386310577})

In [46]:
private_model_eval_dict

{0: {'mae': 0.06011953204870224},
 1: {'mae': 0.10049673914909363},
 2: {'mae': 0.11907858401536942},
 3: {'mae': 0.08741424977779388},
 4: {'mae': 0.06530823558568954},
 5: {'mae': 0.06348166614770889},
 6: {'mae': 0.05634263530373573},
 7: {'mae': 0.06828545033931732},
 8: {'mae': 0.09305131435394287},
 9: {'mae': 0.06849749386310577}}

In [37]:
MODEL_CONFIG

{'_model': ts_inverse.models.fcn.FCN_Predictor,
 'hidden_size': 64,
 '_attack_step_multiplier': 1}

In [ ]:
history

History (loss, distributed):
	round 1: 0.014502647483431715
History (metrics, distributed, evaluate):
{'mae': [(1, 0.08340733835951307)]}

In [7]:
history.metrics_distributed

{'mae': [(1, 0.08340733835951307)]}

In [ ]:
!netstat -ano | findstr :8080

In [ ]:
# import subprocess
# import time
# import threading

# # Function to continuously read from a process' output
# def read_output(process, label):
#     for line in iter(process.stdout.readline, ''):
#         print(f"{label}: {line}", end='')

# # Function to monitor the processes
# def monitor_processes(server, clients):
#     while True:
#         # Check if server has exited
#         if server.poll() is not None:
#             print("Server has exited unexpectedly. Terminating all processes.")
#             terminate_all_processes(server, clients)
#             break

#         # Check if any client has exited
#         for client in clients:
#             if client.poll() is not None:
#                 print(f"Client {clients.index(client)} has exited unexpectedly. Terminating all processes.")
#                 terminate_all_processes(server, clients)
#                 break

#         # Sleep for a short time to avoid constant polling
#         time.sleep(0.5)

# # Function to terminate all processes
# def terminate_all_processes(server, clients):
#     server.terminate()
#     for client in clients:
#         client.terminate()

# print("Starting server...")
# # Start the server
# server = subprocess.Popen(
#     ["python", "src/server.py"], 
#     stdout=subprocess.PIPE, 
#     stderr=subprocess.STDOUT,
#     text=True,
#     bufsize=1
# )
# threading.Thread(target=read_output, args=(server, "Server"), daemon=True).start()

# # Wait for the server to initialize
# time.sleep(3)
# print("Server started.")

# # Start clients
# clients = []
# for i in range(2):
#     print(f"Starting client {i}...")
#     client = subprocess.Popen(
#         ["python", "src/client.py", f"--partition={i}", "--use_cuda=True"], 
#         stdout=subprocess.PIPE, 
#         stderr=subprocess.STDOUT,
#         text=True,
#         bufsize=1
#     )
#     threading.Thread(target=read_output, args=(client, f"Client {i}"), daemon=True).start()
#     clients.append(client)
#     print(f"Started client {i}")

# # Start monitoring thread
# monitor_thread = threading.Thread(target=monitor_processes, args=(server, clients))
# monitor_thread.start()

# # Use try-except to handle interrupts
# try:
#     # Wait for the monitor thread to finish
#     monitor_thread.join()
# except KeyboardInterrupt:
#     print("Terminating all processes due to KeyboardInterrupt...")
#     terminate_all_processes(server, clients)
#     print("Terminated all processes.")